# Imports

In [1]:
# Local application/library specific imports
from pygrex.config import cfg
from pygrex.data_reader import DataReader, GroupInteractionHandler
# from pygrex.evaluator import SlidingWindowEvaluator
from pygrex.explain import RuleBasedGroupRecExplainer
from pygrex.models import ALS
from pygrex.recommender import GroupRecommender
from pygrex.utils import AggregationStrategy
from pygrex.evaluator import ExplanationEvaluator

import time
import pandas as pd
import pickle
import os


In [2]:
# Read the ratings file.
data = DataReader(**cfg.data.test)
data.make_consecutive_ids_in_dataset()
data.binarize(binary_threshold=1)

# Read the file with the group ids
group_handler = GroupInteractionHandler(**cfg.data.groups)
available_groups = group_handler.read_groups("groupsWithHighRatings5.txt")
print("✅ Data preparation complete.\n")

# --- Display Data Summary ---
print("--- Data Summary ---")
print(f"👥 Unique Users: {data.num_user:,}")
print(f"📦 Unique Items: {data.num_item:,}")
print(f"⭐ Total Ratings: {len(data.get_raw_dataset()):,}")
print(f"👨‍👩‍👧‍👦 Number of Groups: {len(available_groups):,}")
print("\nProcessed Ratings DataFrame Head:")
display(data.dataset.head())

✅ Data preparation complete.

--- Data Summary ---
👥 Unique Users: 610
📦 Unique Items: 9,724
⭐ Total Ratings: 100,836
👨‍👩‍👧‍👦 Number of Groups: 17

Processed Ratings DataFrame Head:


,userId,itemId,rating,timestamp
0,0,0,1,964982703
1,0,2,1,964981247
2,0,5,1,964982224
3,0,43,1,964983815
4,0,46,1,964982931


## Step 2: Model Training & Evaluation

With the data prepared, we now select and train a recommendation model. We will use **Alternating Least Squares (ALS)**, a matrix factorization technique for implicit feedback. After training, we will evaluate its performance using a train/test split to measure its Hit Ratio and NDCG.

In [3]:
print("--- 2.1 Model Training ---")

# Train the recommendation model
model = ALS(**cfg.model.als)

# Train the model
start_time = time.time()
model.fit(data)
end_time = time.time()
training_time = end_time - start_time

print(f"✅ Model trained successfully in {training_time:.2f} seconds!")

--- 2.1 Model Training ---


c:\Users\usuar\miniconda3\envs\pygrex-exp-grs\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/10 [00:00<?, ?it/s]

✅ Model trained successfully in 1.00 seconds!


In [ ]:
print("\n--- 2.2 Offline Model Evaluation ---")
# For evaluation, a new model instance must be created.
# The evaluation function handles its own internal data splitting and training.
eval_model = ALS(**cfg.model.als)

# Define evaluation parameters
test_size = 0.2
top_n = 10

print(f"Running evaluation with a {test_size*100:.0f}% test split (Top-{top_n})...")

# Run the evaluation
evaluation_scores = run_evaluation_with_proper_split(
    data_reader=data,
    model=eval_model,
    test_size=test_size,
    top_n=top_n,
)

# Display evaluation results
print("\n--- Evaluation Results ---")
print(f"Hit Ratio @{top_n}: {evaluation_scores.get('Hit Ratio', 0.0):.2%}")
print(f"NDCG @{top_n}: {evaluation_scores.get('NDCG', 0.0):.4f}")
print(f"Evaluation Time: {evaluation_scores.get('evaluation_time', 0):.1f}s")

## Step 3: Group Recommendation

Now that we have a trained model, we can generate recommendations for a group. We will select a group, choose an aggregation strategy to combine individual member preferences, and generate a Top-10 list of recommended items.

In [4]:
print("--- 3. Group Recommendation ---")

# Select a group and strategy
selected_group_id = available_groups[0]  # Let's use the first group as an example
group_members = group_handler.parse_group_members(selected_group_id)
aggregation_strategy = AggregationStrategy.AVG_PREDICTIONS # Use the simple average strategy
top_k = 10

print(f"Generating Top-{top_k} recommendations for group: {selected_group_id}")
print(f"👥 Group Members: {group_members}")
print(f"📊 Aggregation Strategy: {aggregation_strategy.name}")

# --- Generate Recommendations ---
# 1. Instantiate the GroupRecommender
group_recommender = GroupRecommender(data=data)

# 2. Setup the recommendation process
group_recommender.setup_recommendation(
    model=model,
    members=group_members,  # type: ignore
    data=data,
    aggregation_strategy=aggregation_strategy,
                )


# 3. Get the final recommendation list
recommended_items = group_recommender.get_group_recommendations(top_k=top_k)
recommendation_scores = group_recommender.get_recommendation_scores()

print("\n✅ Recommendations generated successfully!")

# --- Display Results ---
rec_data = [
    {
        "Rank": i + 1,
        "Item ID": item_id,
        "Aggregated Score": recommendation_scores.get(item_id, 0.0),
    }
    for i, item_id in enumerate(recommended_items)  # type: ignore
]

rec_df = pd.DataFrame(rec_data)
print(f"\nTop {top_k} Recommended Items:")
display(rec_df)

--- 3. Group Recommendation ---
Generating Top-10 recommendations for group: 522_385_234_452_594
👥 Group Members: [522, 385, 234, 452, 594]
📊 Aggregation Strategy: AVG_PREDICTIONS

✅ Recommendations generated successfully!

Top 10 Recommended Items:


,Rank,Item ID,Aggregated Score
0,1,543,4.636274
1,2,757,4.582981
2,3,564,4.504107
3,4,441,4.488708
4,5,379,4.341830
5,6,475,4.279482
6,7,43,4.268454
7,8,19,4.225248
8,9,748,4.178329
9,10,64,4.147735


## Step 4: Explanation (EXPGRS)

Finally, we generate an explanation for one of the recommendations. We will use the **EXPGRS** method to find a ruled based explanation. This method calculates the Model Fidelity: the percentage of the Top-N list that can be explained by pre-computed association rules from cached files.


In [8]:
print("--- 4. Rule based Explanation (EXPGRS) ---")


def load_cached_data_rules(min_support, min_confidence, rating_threshold):
    """
    Loads pre-computed association rules from the cached_rules folder.
    Returns the loaded object (typically a dict with key "rules") if found, None otherwise.
    Searches several common locations to be robust in notebooks.
    """
    from pathlib import Path

    filename = f"rules_sup{min_support:.2f}_conf{min_confidence:.1f}_rating{rating_threshold:.0f}"
    possible_extensions = [".pkl", ".pickle", ".json"]

    cwd = Path.cwd()
    search_dirs = [
        cwd / "cached_rules",                # current working directory
        cwd.parent / "cached_rules",         # parent (useful when running from notebooks/)
        Path(__file__).resolve().parent / "cached_rules" if '__file__' in globals() else None,  # script dir if available
    ]
    search_dirs = [p for p in search_dirs if p is not None]

    tried_paths = []
    for base in search_dirs:
        for ext in possible_extensions:
            filepath = base / f"{filename}{ext}"
            tried_paths.append(str(filepath))
            if filepath.exists():
                try:
                    if ext in [".pkl", ".pickle"]:
                        with open(filepath, "rb") as f:
                            return pickle.load(f)
                    elif ext == ".json":
                        import json
                        with open(filepath, "r") as f:
                            return json.load(f)
                except Exception as e:
                    print(f"Error loading cached rules from {filepath}: {e}")
                    continue

    print("Cached rules not found. Tried paths:")
    for p in tried_paths:
        print(" -", p)
    return None


def get_user_history(rating_threshold):
    """
    Generates the user interaction history based only on the rating threshold.
    The keys of the returned dictionary are the ORIGINAL user IDs.
    """
    df_filtered = data.dataset[data.dataset["rating"] >= rating_threshold]

    # Group by the 'userId' column (which contains the new, consecutive IDs)
    history_by_new_id = df_filtered.groupby("userId")["itemId"].apply(set).to_dict()

    # Create the final dictionary mapping original user IDs to sets of new item IDs
    history_by_original_id = {}
    for new_id, item_set in history_by_new_id.items():
        try:
            original_id = data.get_original_user_id(int(new_id))
            # The explainer needs the item IDs to be strings to match the rules
            history_by_original_id[original_id] = {str(item) for item in item_set}
        except (ValueError, KeyError):
            continue

    return history_by_original_id

# ----------------------------------------------------------------------- #

min_support = 0.1
min_confidence = 0.1
rating_threshold = 1
minimum_members = 1

# Load cached rules (no Streamlit dependencies)
expected_filename = f"rules_sup{min_support:.2f}_conf{min_confidence:.1f}_rating{rating_threshold:.0f}"
cached_data_rules = load_cached_data_rules(min_support, min_confidence, rating_threshold)
if cached_data_rules is None:
    print("⚠️ Cached rules not found.")
    print("Looked for:", ", ".join(
        [os.path.join("cached_rules", expected_filename + ext) for ext in [".pkl", ".pickle", ".json"]]
    ))
    raise SystemExit("Please place the cached rules file in the 'cached_rules/' folder.")

# Extract rules from loaded structure
cached_rules = cached_data_rules.get("rules") if isinstance(cached_data_rules, dict) else None
if cached_rules is None:
    raise ValueError(
        "Loaded cached rules file does not contain a 'rules' key. Check the file format."
    )

# Get user history
user_history = get_user_history(rating_threshold)

# Create explainer with cached rules
explainer = RuleBasedGroupRecExplainer(
    rules=cached_rules,
    data=data,
    pool_recommendations=recommended_items,
    members=group_members,
    user_history=user_history,
    min_members_threshold=minimum_members,
)

# Compute explanations and metrics
fidelity_score = explainer.find_explanation()
advanced_fidelity_score = explainer.compute_group_fidelity_advanced()
explanation_details = explainer.get_explanation_details()

explanation_results = {
    "fidelity": fidelity_score,
    "advanced_fidelity": advanced_fidelity_score,
    "details": explanation_details,
}

# Evaluate results
_evaluator = ExplanationEvaluator()
metrics = _evaluator.evaluate(explanation_results, explainer_type="EXPGRS")

print("Explanation Fidelity:")
print(f"{metrics.get('fidelity', 0.0):.2%}")
print("-" * 20)

print("Advanced Explanation Fidelity:")
print(f"{explanation_results.get('advanced_fidelity', 0.0):.2%}")
print("-" * 20)

print("Explanation Diversity (GILD):")
print(f"{metrics.get('gild', 0.0):.4f}")
print("-" * 20)


--- 4. Rule based Explanation (EXPGRS) ---
Explanation Fidelity:
10.00%
--------------------
Advanced Explanation Fidelity:
0.00%
--------------------
Explanation Diversity (GILD):
0.0000
--------------------
